# Model 1 v3 · Notebook 2 · Meeting-notes student

Trains the Model 1 v3 student (repo `PLAN.md` §10) from the cached teacher outputs of
Notebook 1; no teacher is ever loaded here.

**Student (~21M).** Each sentence (≤ 64 subwords) is encoded on its own by the sentence encoder
(the 6-layer, d384 encoder pretrained with MLM in the v1 notebook, reused as is); its `[CLS]`
state + sentence-position, speaker and speaker-change embeddings feeds a new 2-layer **meeting
encoder** over up to 256 sentences (longer meetings: windows overlapping by 64). Heads per
sentence: importance, zone (5), Qwen reason (9), `depends_on` pointer to an earlier sentence;
per token: owner / deadline BIO spans.

**Runs** (corpus v1.1, three seeds each): **E4** = gold + RoBERTa-Large KL + Qwen3-0.6B soft
scores (KL), reasons and links; **E1** = gold labels only (the control). E4 runs only if Qwen
passed the quality gate. After training, decision / action-item thresholds are tuned on AMI
validation (zone calibration).

**Inputs:** the `meeting-notes-extractive-v1` dataset (**v1.1**); the MLM encoder and tokenizer
(`checkpoints/mlm/encoder.keras` + `checkpoints/model1_tokenizer/`, from the v1 `model1_scorer`
notebook or a dataset holding them); the output of Notebook 1 on v1.1 (`data/labels/model1_v3/`);
optionally a previous output of this notebook (resume).
**Accelerator:** GPU T4 x2. **Outputs:** `checkpoints/model1_v3_*/seed_*/best.keras`,
`results/model1_v3_*/seed_*/` (metrics, test predictions, history), a preview table and
`manifest_model1_notes.json`.

Smoke mode (`SMOKE=1`, CPU, repo tests): tiny corpus, tokenizer trained here, fake teacher
tables, a tiny model.

In [ ]:
import os

os.environ["KERAS_BACKEND"] = "jax"  # before any keras import
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

import sys
import time
from pathlib import Path

SESSION_START = time.time()
ON_KAGGLE = Path("/kaggle/input").exists()
SMOKE = os.environ.get("SMOKE") == "1"  # tiny CPU run with fake teacher outputs (< 2 min)
WORK = Path(os.environ.get("WORK_DIR", "/kaggle/working"))
INPUT = Path(os.environ.get("INPUT_DIR", "/kaggle/input"))
STAGE = "model1_notes"
print(f"on_kaggle={ON_KAGGLE} smoke={SMOKE} work={WORK}")

In [ ]:
# Exact versions for the training environment.
import subprocess

if ON_KAGGLE:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "jax[cuda12]==0.10.2",
                    "keras==3.15.1", "tokenizers==0.23.3", "pyarrow"], check=True)

## Configuration
The repo's `model1_v3.yaml` and its `_E4` / `_E1` variants.

In [ ]:
CONFIG = {
    "seed": 0,
    "max_hours": 11,                      # per Kaggle session (12 h cap), all runs included
    "min_hours_to_start": 0.5,            # do not start a run with less time left
    "base": {
        "max_hours": 11,
        "corpus": {"revision": "v1.0", "text_field": "sentence_text",
                   "sources": ["synthetic", "synthetic_hn", "ami"]},
        "data": {"max_sentence_tokens": 64, "max_sentences": 256, "window_overlap": 64,
                 "max_meetings": None, "sentence_buckets": [32, 64, 128, 256],
                 "hard_negative_oversample": 2},
        "model": {"vocab_size": 16000, "d_model": 384, "num_layers": 6, "meeting_layers": 2,
                  "num_heads": 6, "ffn_dim": 1536, "max_len": 512, "num_speakers": 8,
                  "dropout": 0.1, "init_std": 0.02, "num_zones": 5, "num_reasons": 9,
                  "num_span_tags": 5},
        "loss": {"importance": 1.0, "zone": 0.5, "spans": 0.3, "qwen_kl": 0.5, "roberta_kl": 0.5,
                 "reason": 0.3, "depends_on": 0.3, "temperature": 2.0,
                 "zone_class_weights": "inverse_sqrt"},
        "optimizer": {"name": "adamw", "lr_sentence_encoder": 1.0e-4, "lr_meeting": 5.0e-4,
                      "warmup_fraction": 0.1, "schedule": "cosine", "weight_decay": 0.01},
        "inference": {"zone_calibration": "balanced", "zone_threshold_grid": [0.005, 0.5, 0.005]},
        "training": {"sentences_per_batch": 2048, "epochs": 15, "max_steps": None,
                     "mixed_precision": "mixed_float16",
                     "data_parallel": False,  # one T4: NCCL across 2x T4 failed (2026-10)
                     "early_stopping": {"monitor": "val_loss", "mode": "min", "patience": 2}},
    },
    "variants": {
        "model1_v3_E4": {"teachers": ["qwen", "roberta"]},
        "model1_v3_E1": {"teachers": [], "loss": {"qwen_kl": 0.0, "roberta_kl": 0.0,
                                                  "reason": 0.0, "depends_on": 0.0}},
        # Gold + RoBERTa only: isolates the strong teacher from Qwen (approved 2026-10-25).
        "model1_v3_ER": {"teachers": ["roberta"], "loss": {"qwen_kl": 0.0, "reason": 0.0,
                                                          "depends_on": 0.0}},
    },
    # Corpus v1.1 (2026-10-25): E4 (best on v1.0) and the E1 control, three seeds each,
    # interleaved so the comparison exists after the first two runs.
    "runs": [["model1_v3_E4", 0], ["model1_v3_E1", 0], ["model1_v3_E4", 1], ["model1_v3_E1", 1],
             ["model1_v3_E4", 2], ["model1_v3_E1", 2]],
}
if SMOKE:
    CONFIG["max_hours"] = CONFIG["base"]["max_hours"] = 0.2
    CONFIG["min_hours_to_start"] = 0.0
    CONFIG["runs"] = [["model1_v3_E4", 0], ["model1_v3_E1", 0]]
    CONFIG["base"]["data"].update(max_sentence_tokens=16, max_sentences=32, window_overlap=8,
                                  sentence_buckets=[8, 16, 32])
    CONFIG["base"]["model"].update(d_model=32, num_layers=1, meeting_layers=1, num_heads=2,
                                   ffn_dim=64, max_len=16)
    CONFIG["base"]["training"].update(sentences_per_batch=64, epochs=1, max_steps=4,
                                      mixed_precision="float32")

In [ ]:
import json
import math
import random
import re
import shutil
import subprocess
from collections.abc import Callable, Sequence
from dataclasses import dataclass
from typing import Any

import numpy as np
import pyarrow as pa
import pyarrow.parquet as pq

random.seed(CONFIG["seed"])
np.random.seed(CONFIG["seed"])


def json_dump(path, data):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(data, indent=2, default=str) + "\n")


def hours_left():
    return CONFIG["max_hours"] - (time.time() - SESSION_START) / 3600


def find_dirs(root, predicate):
    """Directories under ``root`` (following links) for which ``predicate(dir)`` holds."""
    found = []
    if root.exists():
        for d, _, _ in os.walk(root, followlinks=True):
            if predicate(Path(d)):
                found.append(Path(d))
    return sorted(found, key=lambda p: len(p.parts))


def find_corpus():
    """The attached corpus release (``data/{synthetic,ami}/sentences/*.parquet``)."""
    found = find_dirs(INPUT, lambda d: (d / "data/synthetic/sentences/train.parquet").exists())
    if not found:
        raise SystemExit("Attach the meeting-notes-extractive-v1 dataset (Add data).")
    return found[0]

import logging

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(message)s")
log = logging.getLogger(__name__)
PROCESSED = WORK / "data" / "processed" / "model1_v3"
CKPT, RESULTS = WORK / "checkpoints", WORK / "results"


def commit_sha():
    return None  # the notebook is not a git checkout


# Resume: a previous output of this notebook (checkpoints, results, arrays).
for prev in find_dirs(INPUT, lambda d: (d / f"manifest_{STAGE}.json").exists()):
    for sub in ("data", "checkpoints", "results"):
        if (prev / sub).exists():
            shutil.copytree(prev / sub, WORK / sub, dirs_exist_ok=True)
    print("restored from", prev)

## 1. Inputs
Corpus, tokenizer and MLM encoder (from the v1 notebook's output), teacher tables and gate
(from Notebook 1). Smoke runs make all of them here.

In [ ]:
from __future__ import annotations


from tokenizers import Tokenizer


SPLITS = ("train", "validation", "test")


SOURCES = ("synthetic", "synthetic_hn", "ami")


MEETING_SENTENCES = [
    ("Ana", "Thanks for joining, this is the weekly sync.", 0, "context", False),
    ("Ben", "The load tests failed again on Monday.", 1, "discussion", False),
    ("Ana", "Okay, we hold the release until they pass.", 1, "decision", False),
    ("Ben", "Ben will fix the test rig by Friday.", 1, "action_item", True),
    ("Cleo", "Sounds good.", 0, "discussion", False),
    ("Ana", "See you next week.", 0, "closing", False),
]


def _spans(text: str) -> list[dict[str, Any]]:
    """Owner ("Ben", at the start) and deadline ("Friday") spans of the action-item sentence."""
    day = text.index("Friday")
    return [
        {"start": 0, "end": 3, "role": "owner"},
        {"start": day, "end": day + len("Friday"), "role": "deadline"},
    ]


def tiny_corpus_rows(
    note_id: str, split: str, source: str = "synthetic", repeat: int = 1
) -> list[dict[str, Any]]:
    """Rows of one meeting in the corpus ``sentences`` schema (``repeat`` x 6 sentences)."""
    ami = source == "ami"
    rows = []
    for k in range(repeat):
        for j, (speaker, text, importance, zone, has_spans) in enumerate(MEETING_SENTENCES):
            if ami and zone in ("context", "closing"):
                zone = "discussion"  # AMI has no context / closing annotation
            rows.append(
                {
                    "note_id": note_id,
                    "source": source,
                    "meeting_type": "ami_design" if ami else "standup",
                    "domain": "product_design" if ami else "engineering",
                    "split": split,
                    "sentence_id": k * len(MEETING_SENTENCES) + j,
                    "speaker": speaker,
                    "sentence_text": text,
                    "sentence_text_asr": text.lower().replace(",", "").replace(".", ""),
                    "importance": importance,
                    "confidence": 0.8 if ami else 1.0,
                    "zone": zone,
                    "segment_id": None,
                    "family_id": None,
                    "hard_negative": source == "synthetic_hn",
                    "token_count": len(text.split()),
                    "spans": _spans(text) if has_spans and not ami else [],
                }
            )
    return rows


def tiny_note_row(note_id: str, split: str, rows: list[dict[str, Any]]) -> dict[str, Any]:
    """The ``notes`` row of a meeting (AMI rows carry summary sections)."""
    ami = rows[0]["source"] == "ami"
    return {
        "note_id": note_id,
        "source": rows[0]["source"],
        "meeting_type": rows[0]["meeting_type"],
        "split": split,
        "n_sentences": len(rows),
        "n_words": sum(r["token_count"] for r in rows),
        "n_participants": len({r["speaker"] for r in rows}),
        "summary_abstract": "The team discussed the failing load tests." if ami else None,
        "summary_decisions": "The release is held until the tests pass." if ami else None,
        "summary_actions": "Ben fixes the test rig by Friday." if ami else None,
        "summary_problems": "The load tests fail." if ami else None,
    }


def write_tiny_corpus(corpus_dir: str | Path, meetings_per_split: int = 3) -> None:
    """Write the release layout; meeting ``i`` has ``6 * (1 + i % 6)`` sentences."""
    for split in SPLITS:
        layers: dict[str, tuple[list[dict[str, Any]], list[dict[str, Any]]]] = {
            "synthetic": ([], []),
            "ami": ([], []),
        }
        for i in range(meetings_per_split):
            source = SOURCES[i % 3]
            rows = tiny_corpus_rows(f"{split}-{i}", split, source, repeat=1 + i % 6)
            sentences, notes = layers["ami" if source == "ami" else "synthetic"]
            sentences += rows
            notes.append(tiny_note_row(f"{split}-{i}", split, rows))
        for layer, (sentences, notes) in layers.items():
            for table, records in (("sentences", sentences), ("notes", notes)):
                out = Path(corpus_dir) / "data" / layer / table
                out.mkdir(parents=True, exist_ok=True)
                pq.write_table(pa.Table.from_pylist(records), out / f"{split}.parquet")


PAD_ID, UNK_ID, CLS_ID, SEP_ID, MASK_ID = 0, 1, 2, 3, 4


NUM_SPEAKERS = 8


SPECIAL_TOKENS = ["[PAD]", "[UNK]", "[CLS]", "[SEP]", "[MASK]"] + [
    f"[S{i}]" for i in range(1, NUM_SPEAKERS + 1)
]


def check_special_tokens(tokenizer: Tokenizer) -> None:
    """Raise ``ValueError`` if special tokens do not have their fixed IDs."""
    for expected_id, token in enumerate(SPECIAL_TOKENS):
        actual = tokenizer.token_to_id(token)
        if actual != expected_id:
            raise ValueError(f"{token} has ID {actual}, expected {expected_id}")


def write_jsonl(path: str | Path, records: Iterable[dict[str, Any]]) -> int:
    """Write one JSON object per line (UTF-8, non-ASCII kept). Returns the number written."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    n = 0
    with path.open("w", encoding="utf-8") as f:
        for record in records:
            f.write(json.dumps(record, ensure_ascii=False) + "\n")
            n += 1
    return n


ZONES = ("context", "discussion", "decision", "action_item", "closing")


SPAN_TAGS = ("O", "B-owner", "I-owner", "B-deadline", "I-deadline")


DECISION, ACTION = ZONES.index("decision"), ZONES.index("action_item")


#: Qwen teacher reason classes (PLAN.md 10.3); index = the student's ``reason`` head output.
REASONS = (
    "agreed_outcome",
    "commitment_with_owner",
    "risk_or_blocker",
    "key_fact",
    "status_update",
    "rejected_proposal",
    "tentative",
    "question",
    "filler",
)


def window_starts(length: int, size: int, overlap: int) -> list[int]:
    """Start offsets of windows of ``size`` sentences overlapping by ``overlap``.

    Windows advance by ``size - overlap``; the last one is aligned to the end of the meeting so
    every window except a short meeting's single window is full.
    """
    if not 0 <= overlap < size:
        raise ValueError(f"window_overlap must be in [0, {size}); got {overlap}")
    if length <= size:
        return [0]
    starts = list(range(0, length - size, size - overlap))
    return [*starts, length - size]


IGNORE = -1


COLUMNS = (
    "note_id",
    "source",
    "sentence_id",
    "speaker",
    "importance",
    "confidence",
    "zone",
    "hard_negative",
    "spans",
)


def layer_of(source: str) -> str:
    """Release layer holding a source: ``synthetic`` (incl. hard negatives) or ``ami``."""
    return "ami" if source == "ami" else "synthetic"


def corpus_split_paths(corpus_dir: str | Path, split: str, sources: Sequence[str]) -> list[Path]:
    """Sentence tables of ``split`` for the layers of ``sources`` (synthetic first, then AMI)."""
    unknown = set(sources) - set(SOURCES)
    if unknown:
        raise ValueError(f"Unknown sources {sorted(unknown)}; expected a subset of {SOURCES}")
    layers = sorted({layer_of(s) for s in sources}, key=("synthetic", "ami").index)
    paths = [
        Path(corpus_dir) / "data" / layer / "sentences" / f"{split}.parquet" for layer in layers
    ]
    missing = [p for p in paths if not p.exists()]
    if missing:
        raise FileNotFoundError(f"Corpus split not found: {', '.join(map(str, missing))}")
    return paths


def read_corpus_split(
    corpus_dir: str | Path, split: str, text_field: str, sources: Sequence[str]
) -> dict[str, list[dict[str, Any]]]:
    """Rows of one split grouped by meeting, sentences sorted by ``sentence_id``.

    Meetings keep the order of their first row (synthetic layer first). Rows whose ``source``
    is not in ``sources`` are dropped.
    """
    meetings: dict[str, list[dict[str, Any]]] = {}
    for path in corpus_split_paths(corpus_dir, split, sources):
        for row in pq.read_table(path, columns=[*COLUMNS, text_field]).to_pylist():
            if row["source"] in sources:
                row["text"] = row.pop(text_field)
                meetings.setdefault(row["note_id"], []).append(row)
    for rows in meetings.values():
        rows.sort(key=lambda r: r["sentence_id"])
    return meetings


def sentence_rows(dataset: NotesSet) -> dict[tuple[str, int], int]:
    """``(note_id, sentence_id) -> row`` of the sentence arrays (to align teacher outputs)."""
    rows: dict[tuple[str, int], int] = {}
    sentence_id = np.asarray(dataset["sentence_id"])
    starts, lengths = np.asarray(dataset["note_start"]), np.asarray(dataset["note_length"])
    for note_id, start, n in zip(dataset.note_ids, starts, lengths, strict=True):
        for row in range(int(start), int(start) + int(n)):
            rows[(note_id, int(sentence_id[row]))] = row
    return rows


def speaker_numbers(speakers: Sequence[str]) -> tuple[list[int], int]:
    """1-based speaker numbers by first appearance, capped at ``NUM_SPEAKERS``.

    Returns the numbers and how many distinct speakers were capped.
    """
    order: dict[str, int] = {}
    for s in speakers:
        order.setdefault(s, len(order) + 1)
    capped = max(0, len(order) - NUM_SPEAKERS)
    return [min(order[s], NUM_SPEAKERS) for s in speakers], capped


def span_bio(
    offsets: Sequence[tuple[int, int]], spans: Sequence[dict[str, Any]] | None
) -> list[int]:
    """BIO tag index (:data:`SPAN_TAGS`) for each subword, from character spans.

    A subword belongs to a span when its character range overlaps it; the first such subword
    is ``B-<role>``, the following ones ``I-<role>``.
    """
    tags = [0] * len(offsets)
    for span in spans or ():
        begin = SPAN_TAGS.index(f"B-{span['role']}")
        inside = False
        for i, (start, end) in enumerate(offsets):
            if start < span["end"] and end > span["start"]:
                tags[i] = begin + 1 if inside else begin
                inside = True
    return tags


def encode_sentence(
    tokenizer: Tokenizer, text: str, spans: Sequence[dict[str, Any]] | None, max_tokens: int
) -> tuple[np.ndarray, np.ndarray, bool]:
    """``[CLS] subwords [SEP]`` padded to ``max_tokens``, its span tags and a truncation flag."""
    enc = tokenizer.encode(text, add_special_tokens=False)
    budget = max_tokens - 2
    ids, tags = list(enc.ids), span_bio(enc.offsets, spans)
    truncated = len(ids) > budget
    ids, tags = ids[:budget], tags[:budget]
    token_ids = np.full(max_tokens, PAD_ID, dtype=np.int32)
    span_tags = np.full(max_tokens, IGNORE, dtype=np.int8)
    token_ids[: len(ids) + 2] = [CLS_ID, *ids, SEP_ID]
    span_tags[1 : len(ids) + 1] = tags
    return token_ids, span_tags, truncated


def encode_split(
    tokenizer: Tokenizer, meetings: dict[str, list[dict[str, Any]]], data_cfg: dict[str, Any]
) -> tuple[dict[str, np.ndarray], list[str], dict[str, int]]:
    """Arrays for one split (see module docstring), the meeting IDs and counts."""
    max_tokens, size = data_cfg["max_sentence_tokens"], data_cfg["max_sentences"]
    note_ids = list(meetings)
    n = sum(len(rows) for rows in meetings.values())
    a: dict[str, np.ndarray] = {
        "token_ids": np.zeros((n, max_tokens), dtype=np.int32),
        "span_tags": np.zeros((n, max_tokens), dtype=np.int8),
        "importance": np.zeros(n, dtype=np.int8),
        "confidence": np.zeros(n, dtype=np.float32),
        "zone": np.zeros(n, dtype=np.int8),
        "speaker": np.zeros(n, dtype=np.int8),
        "sentence_id": np.zeros(n, dtype=np.int32),
        "hard_negative": np.zeros(n, dtype=bool),
        "note_start": np.zeros(len(note_ids), dtype=np.int64),
        "note_length": np.zeros(len(note_ids), dtype=np.int32),
        "source": np.zeros(len(note_ids), dtype=np.int8),
    }
    windows: list[tuple[int, int, int]] = []
    counts = {"truncated_sentences": 0, "meetings_speakers_capped": 0}
    row = 0
    for m, note_id in enumerate(note_ids):
        rows = meetings[note_id]
        speakers, capped = speaker_numbers([r["speaker"] for r in rows])
        counts["meetings_speakers_capped"] += bool(capped)
        a["note_start"][m], a["note_length"][m] = row, len(rows)
        a["source"][m] = SOURCES.index(rows[0]["source"])
        for r, speaker in zip(rows, speakers, strict=True):
            ids, tags, truncated = encode_sentence(tokenizer, r["text"], r["spans"], max_tokens)
            a["token_ids"][row], a["span_tags"][row] = ids, tags
            a["importance"][row] = r["importance"]
            a["confidence"][row] = r["confidence"]
            a["zone"][row] = ZONES.index(r["zone"])
            a["speaker"][row] = speaker
            a["sentence_id"][row] = r["sentence_id"]
            a["hard_negative"][row] = r["hard_negative"]
            counts["truncated_sentences"] += truncated
            row += 1
        for start in window_starts(len(rows), size, data_cfg["window_overlap"]):
            windows.append((m, start, min(size, len(rows) - start)))
    w = np.asarray(windows, dtype=np.int32).reshape(-1, 3)
    a["window_note"], a["window_offset"], a["window_length"] = w[:, 0], w[:, 1], w[:, 2]
    counts = {"meetings": len(note_ids), "sentences": n, "windows": len(w), **counts}
    return a, note_ids, counts


@dataclass
class NotesSet:
    """Encoded split (memory-mapped arrays, see module docstring)."""

    note_ids: list[str]
    arrays: dict[str, np.ndarray]

    @classmethod
    def load(cls, directory: str | Path) -> NotesSet:
        d = Path(directory)
        arrays = {p.stem: np.load(p, mmap_mode="r") for p in sorted(d.glob("*.npy"))}
        return cls(json.loads((d / "note_ids.json").read_text()), arrays)

    def __getitem__(self, name: str) -> np.ndarray:
        return self.arrays[name]

    @property
    def num_windows(self) -> int:
        return len(self.arrays["window_note"])


def bucket_for(length: int, buckets: Sequence[int]) -> int:
    """Smallest bucket that holds ``length`` sentences."""
    for b in sorted(buckets):
        if length <= b:
            return b
    raise ValueError(f"{length} sentences exceed the largest bucket {max(buckets)}")


def gather_windows(dataset: NotesSet, windows: Sequence[int], slots: int) -> dict[str, np.ndarray]:
    """Padded batch for the given window indices, ``slots`` sentences per window.

    Returns ``token_ids`` / ``span_tags`` (B, slots, T), per-sentence ``importance``,
    ``confidence``, ``zone``, ``speaker``, ``hard_negative`` (B, slots), ``sentence_mask``
    (B, slots) bool, ``rows`` (B, slots) int64 sentence rows (``-1`` on padding, to look up
    teacher arrays) and ``source`` (B,). Padded sentences have ``PAD_ID`` tokens, ``IGNORE``
    labels and speaker 0.
    """
    b = len(windows)
    rows = np.full((b, slots), -1, dtype=np.int64)
    for i, w in enumerate(windows):
        length = int(dataset["window_length"][w])
        if length > slots:
            raise ValueError(f"window {w} has {length} sentences > {slots} slots")
        note = int(dataset["window_note"][w])
        start = int(dataset["note_start"][note]) + int(dataset["window_offset"][w])
        rows[i, :length] = np.arange(start, start + length)
    mask = rows >= 0
    safe = np.where(mask, rows, 0)

    def take(name: str, fill: float | int) -> np.ndarray:
        values = np.asarray(dataset[name])[safe]
        pad = mask.reshape(mask.shape + (1,) * (values.ndim - 2))
        return np.where(pad, values, np.asarray(fill, dtype=values.dtype))

    return {
        "token_ids": take("token_ids", PAD_ID),
        "span_tags": take("span_tags", IGNORE),
        "importance": take("importance", IGNORE),
        "confidence": take("confidence", 0.0),
        "zone": take("zone", IGNORE),
        "speaker": take("speaker", 0),
        "hard_negative": take("hard_negative", False),
        "sentence_mask": mask,
        "rows": rows,
        "source": np.asarray(dataset["source"])[np.asarray(dataset["window_note"])[list(windows)]],
    }


def build_split(
    tokenizer: Tokenizer, cfg: dict[str, Any], split: str, out_dir: str | Path
) -> dict[str, int]:
    """Read, encode and save one split. Returns the counts written to ``meta.json``."""
    corpus = cfg["corpus"]
    meetings = read_corpus_split(
        cfg["paths"]["corpus_dir"], split, corpus["text_field"], corpus["sources"]
    )
    cap = cfg["data"]["max_meetings"]
    if cap is not None:
        meetings = dict(list(meetings.items())[:cap])
    arrays, note_ids, counts = encode_split(tokenizer, meetings, cfg["data"])
    out = Path(out_dir)
    out.mkdir(parents=True, exist_ok=True)
    for name, array in arrays.items():
        np.save(out / f"{name}.npy", array)
    (out / "note_ids.json").write_text(json.dumps(note_ids))
    (out / "meta.json").write_text(json.dumps(counts, indent=2) + "\n")
    return counts

In [ ]:
from tokenizers import Tokenizer


def find_file(rel):
    found = find_dirs(INPUT, lambda d: (d / rel).exists())
    return found[0] / rel if found else None


def smoke_tokenizer(corpus):
    """Smoke runs only: a tiny WordPiece with the fixed special-token IDs."""
    from tokenizers import BertWordPieceTokenizer

    texts = [r["sentence_text"] for layer in ("synthetic", "ami")
             for r in pq.read_table(corpus / f"data/{layer}/sentences/train.parquet").to_pylist()]
    wp = BertWordPieceTokenizer(lowercase=True, strip_accents=True)
    wp.train_from_iterator(texts, vocab_size=300, min_frequency=1, special_tokens=SPECIAL_TOKENS,
                           show_progress=False)
    return wp._tokenizer


def write_fake_teachers(corpus, teacher_dir):
    """Smoke runs only: Qwen / RoBERTa tables in the Notebook 1 format and a passed gate."""
    rng = np.random.default_rng(0)
    for split in SPLITS:
        rows = [r for layer in ("synthetic", "ami") for r in
                pq.read_table(corpus / f"data/{layer}/sentences/{split}.parquet").to_pylist()]
        qwen = [{"note_id": r["note_id"], "sentence_id": r["sentence_id"],
                 "score": float(rng.random()), "raw_score": 1,
                 "reason": REASONS[r["sentence_id"] % len(REASONS)],
                 "depends_on": r["sentence_id"] - 1} for r in rows]
        rob = [{"note_id": r["note_id"], "sentence_id": r["sentence_id"],
                "logit": float(rng.normal())} for r in rows]
        for name, table in (("qwen", qwen), ("roberta", rob)):
            (teacher_dir / name).mkdir(parents=True, exist_ok=True)
            pq.write_table(pa.Table.from_pylist(table), teacher_dir / name / f"{split}.parquet")
    json_dump(teacher_dir / "qwen" / "gate.json", {"passed": True, "smoke": True})


if SMOKE:
    CORPUS = WORK / "tiny_corpus"
    write_tiny_corpus(CORPUS, meetings_per_split=6)
    TOK_FILE = WORK / "tokenizer" / "tokenizer.json"
    TOK_FILE.parent.mkdir(parents=True, exist_ok=True)
    smoke_tokenizer(CORPUS).save(str(TOK_FILE))
    MLM_ENCODER = None
    TEACHER_DIR = WORK / "smoke_labels"
    write_fake_teachers(CORPUS, TEACHER_DIR)
else:
    CORPUS = find_corpus()
    TOK_FILE = (find_file("checkpoints/model1_tokenizer/tokenizer.json")
                or find_file("data/tokenizer/tokenizer.json"))
    MLM_ENCODER = find_file("checkpoints/mlm/encoder.keras")
    labels = find_dirs(INPUT, lambda d: (d / "qwen/gate.json").exists()
                       or (d / "roberta/train.parquet").exists())
    TEACHER_DIR = labels[0] if labels else None
    if TOK_FILE is None or MLM_ENCODER is None:
        raise SystemExit("Attach the output of the model1_scorer notebook (tokenizer + MLM).")
TOKENIZER = Tokenizer.from_file(str(TOK_FILE))
check_special_tokens(TOKENIZER)
GATE_PATH = None if TEACHER_DIR is None else TEACHER_DIR / "qwen" / "gate.json"
GATE = json.loads(GATE_PATH.read_text()) if GATE_PATH and GATE_PATH.exists() else None
print("corpus:", CORPUS, "| tokenizer:", TOK_FILE, "| MLM:", MLM_ENCODER)
print("teachers:", TEACHER_DIR, "| Qwen gate:", GATE)

## 2. Arrays
One row per sentence (`[CLS] subwords [SEP]`, 64 tokens), meetings stored one after another, windows of ≤ 256 sentences.

In [ ]:
for split in SPLITS:
    if not (PROCESSED / split / "meta.json").exists():
        counts = build_split(TOKENIZER, {"paths": {"corpus_dir": CORPUS}, **CONFIG["base"]}, split,
                             PROCESSED / split)
        print(split, counts)
    else:
        print(split, json.loads((PROCESSED / split / "meta.json").read_text()), "(restored)")

## 3. Model, losses and training (JAX from here on)
The same classes as the repo's `src/` (registered under `dialogue_models`, same arguments), so
the saved `.keras` files load there.

In [ ]:
from __future__ import annotations

import jax
import keras
from keras import ops

jax.config.update("jax_compilation_cache_dir", os.environ.get("JAX_CACHE_DIR", "/tmp/jax_cache"))
jax.config.update("jax_persistent_cache_min_compile_time_secs", 1.0)
print("keras", keras.__version__, "| jax", jax.__version__, "| devices:", jax.devices())
if ON_KAGGLE and not SMOKE:
    assert jax.devices()[0].platform == "gpu", "No GPU: Settings -> Accelerator -> GPU T4 x2."


@keras.saving.register_keras_serializable(package="dialogue_models")
class TokenAndPositionEmbedding(keras.layers.Layer):
    """Sum of token, learned position and (optionally) speaker embeddings, then dropout.

    Args:
        vocab_size: number of token IDs.
        max_len: maximum sequence length (number of learned positions).
        d_model: embedding width.
        num_speakers: number of speakers; 0 disables the speaker embedding. Speaker ID 0 is
            reserved for "no speaker" (padding / special tokens), so the table has
            ``num_speakers + 1`` rows.
        dropout: dropout rate applied to the summed embedding.
        init_std: standard deviation of the truncated-normal initialiser.
    """

    def __init__(
        self,
        vocab_size: int,
        max_len: int,
        d_model: int,
        num_speakers: int = 0,
        dropout: float = 0.1,
        init_std: float = 0.02,
        **kwargs: Any,
    ) -> None:
        super().__init__(**kwargs)
        self.vocab_size = vocab_size
        self.max_len = max_len
        self.d_model = d_model
        self.num_speakers = num_speakers
        self.dropout_rate = dropout
        self.init_std = init_std
        init = keras.initializers.TruncatedNormal(stddev=init_std)
        self.token_embedding = keras.layers.Embedding(
            vocab_size, d_model, embeddings_initializer=init, name="token_embedding"
        )
        self.speaker_embedding = (
            keras.layers.Embedding(
                num_speakers + 1, d_model, embeddings_initializer=init, name="speaker_embedding"
            )
            if num_speakers
            else None
        )
        self.dropout = keras.layers.Dropout(dropout)

    def build(self, input_shape: tuple[int | None, ...]) -> None:
        self.token_embedding.build(input_shape)
        if self.speaker_embedding is not None:
            self.speaker_embedding.build(input_shape)
        self.position_embedding = self.add_weight(
            name="position_embedding",
            shape=(self.max_len, self.d_model),
            initializer=keras.initializers.TruncatedNormal(stddev=self.init_std),
        )
        super().build(input_shape)

    def call(self, token_ids, speaker_ids=None, training: bool | None = None):
        """Embed ``token_ids`` (batch, length); ``speaker_ids`` has the same shape."""
        length = ops.shape(token_ids)[-1]
        x = self.token_embedding(token_ids)
        x = x + ops.cast(self.position_embedding[:length], x.dtype)[None, :, :]
        if self.speaker_embedding is not None:
            if speaker_ids is None:
                raise ValueError("speaker_ids is required when num_speakers > 0")
            x = x + self.speaker_embedding(speaker_ids)
        return self.dropout(x, training=training)

    def compute_output_shape(self, input_shape: tuple[int | None, ...]) -> tuple[int | None, ...]:
        return (*input_shape, self.d_model)

    def get_config(self) -> dict[str, Any]:
        config = super().get_config()
        config.update(
            vocab_size=self.vocab_size,
            max_len=self.max_len,
            d_model=self.d_model,
            num_speakers=self.num_speakers,
            dropout=self.dropout_rate,
            init_std=self.init_std,
        )
        return config


def _dense_init(std: float) -> keras.initializers.Initializer:
    return keras.initializers.TruncatedNormal(stddev=std)


def attention_mask_from_padding(padding_mask):
    """(batch, length) bool -> (batch, length, length) mask: queries attend to real keys."""
    return ops.logical_and(padding_mask[:, :, None], padding_mask[:, None, :])


@keras.saving.register_keras_serializable(package="dialogue_models")
class TransformerEncoderBlock(keras.layers.Layer):
    """Pre-LN block: ``x + MHA(LN(x))`` then ``x + FFN(LN(x))``, with dropout on both branches.

    The FFN is ``Dense(ffn_dim) -> GELU -> Dense(d_model)``.
    """

    def __init__(
        self,
        d_model: int,
        num_heads: int,
        ffn_dim: int,
        dropout: float = 0.1,
        init_std: float = 0.02,
        **kwargs: Any,
    ) -> None:
        super().__init__(**kwargs)
        if d_model % num_heads:
            raise ValueError(f"d_model={d_model} is not divisible by num_heads={num_heads}")
        self.d_model = d_model
        self.num_heads = num_heads
        self.ffn_dim = ffn_dim
        self.dropout_rate = dropout
        self.init_std = init_std
        self.attn_norm = keras.layers.LayerNormalization(epsilon=1e-6, name="attn_norm")
        self.attention = keras.layers.MultiHeadAttention(
            num_heads=num_heads,
            key_dim=d_model // num_heads,
            dropout=dropout,
            kernel_initializer=_dense_init(init_std),
            name="attention",
        )
        self.attn_dropout = keras.layers.Dropout(dropout)
        self.ffn_norm = keras.layers.LayerNormalization(epsilon=1e-6, name="ffn_norm")
        self.ffn_in = keras.layers.Dense(
            ffn_dim, activation="gelu", kernel_initializer=_dense_init(init_std), name="ffn_in"
        )
        self.ffn_out = keras.layers.Dense(
            d_model, kernel_initializer=_dense_init(init_std), name="ffn_out"
        )
        self.ffn_dropout = keras.layers.Dropout(dropout)

    def build(self, input_shape: tuple[int | None, ...]) -> None:
        self.attn_norm.build(input_shape)
        self.attention.build(input_shape, input_shape)
        self.ffn_norm.build(input_shape)
        self.ffn_in.build(input_shape)
        self.ffn_out.build((*input_shape[:-1], self.ffn_dim))
        super().build(input_shape)

    def call(self, x, padding_mask=None, training: bool | None = None):
        """``x``: (batch, length, d_model); ``padding_mask``: (batch, length) bool, True = real."""
        mask = None if padding_mask is None else attention_mask_from_padding(padding_mask)
        h = self.attn_norm(x)
        h = self.attention(h, h, attention_mask=mask, training=training)
        x = x + self.attn_dropout(h, training=training)
        h = self.ffn_out(self.ffn_in(self.ffn_norm(x)))
        return x + self.ffn_dropout(h, training=training)

    def compute_output_shape(self, input_shape: tuple[int | None, ...]) -> tuple[int | None, ...]:
        return input_shape

    def get_config(self) -> dict[str, Any]:
        config = super().get_config()
        config.update(
            d_model=self.d_model,
            num_heads=self.num_heads,
            ffn_dim=self.ffn_dim,
            dropout=self.dropout_rate,
            init_std=self.init_std,
        )
        return config


@keras.saving.register_keras_serializable(package="dialogue_models")
class TransformerEncoder(keras.layers.Layer):
    """Embeddings, ``num_layers`` pre-LN blocks and a final LayerNorm.

    Token ID 0 (``[PAD]``) is treated as padding. Returns (batch, length, d_model).
    """

    def __init__(
        self,
        vocab_size: int,
        max_len: int,
        d_model: int,
        num_layers: int,
        num_heads: int,
        ffn_dim: int,
        dropout: float = 0.1,
        init_std: float = 0.02,
        num_speakers: int = 0,
        pad_id: int = 0,
        **kwargs: Any,
    ) -> None:
        super().__init__(**kwargs)
        self.vocab_size = vocab_size
        self.max_len = max_len
        self.d_model = d_model
        self.num_layers = num_layers
        self.num_heads = num_heads
        self.ffn_dim = ffn_dim
        self.dropout_rate = dropout
        self.init_std = init_std
        self.num_speakers = num_speakers
        self.pad_id = pad_id
        self.embedding = TokenAndPositionEmbedding(
            vocab_size, max_len, d_model, num_speakers, dropout, init_std, name="embedding"
        )
        self.blocks = [
            TransformerEncoderBlock(
                d_model, num_heads, ffn_dim, dropout, init_std, name=f"block_{i}"
            )
            for i in range(num_layers)
        ]
        self.final_norm = keras.layers.LayerNormalization(epsilon=1e-6, name="final_norm")

    def build(self, input_shape: tuple[int | None, ...]) -> None:
        self.embedding.build(input_shape)
        hidden = (*input_shape, self.d_model)
        for block in self.blocks:
            block.build(hidden)
        self.final_norm.build(hidden)
        super().build(input_shape)

    def call(self, token_ids, speaker_ids=None, training: bool | None = None):
        padding_mask = ops.not_equal(token_ids, self.pad_id)
        x = self.embedding(token_ids, speaker_ids, training=training)
        for block in self.blocks:
            x = block(x, padding_mask=padding_mask, training=training)
        return self.final_norm(x)

    def compute_output_shape(self, input_shape: tuple[int | None, ...]) -> tuple[int | None, ...]:
        return (*input_shape, self.d_model)

    def get_config(self) -> dict[str, Any]:
        config = super().get_config()
        config.update(
            vocab_size=self.vocab_size,
            max_len=self.max_len,
            d_model=self.d_model,
            num_layers=self.num_layers,
            num_heads=self.num_heads,
            ffn_dim=self.ffn_dim,
            dropout=self.dropout_rate,
            init_std=self.init_std,
            num_speakers=self.num_speakers,
            pad_id=self.pad_id,
        )
        return config


ENCODER_KEYS = (
    "vocab_size",
    "max_len",
    "d_model",
    "num_layers",
    "num_heads",
    "ffn_dim",
    "dropout",
    "init_std",
    "num_speakers",
)


def load_encoder_weights(student: keras.Model, checkpoint: str | Path) -> None:
    """Copy the encoder of an MLM ``encoder.keras`` into ``student``.

    The architecture must match (dropout may differ: it has no weights).
    """
    pretrained = keras.saving.load_model(checkpoint).get_layer("encoder")
    target = student.get_layer("encoder")
    keys = [k for k in ENCODER_KEYS if k != "dropout"]
    have, want = pretrained.get_config(), target.get_config()
    if diff := {k: (have[k], want[k]) for k in keys if have[k] != want[k]}:
        raise ValueError(f"Encoder architecture mismatch (checkpoint, student): {diff}")
    target.set_weights(pretrained.get_weights())


NEG = -1e9


MODEL_KEYS = (
    "vocab_size",
    "max_len",
    "d_model",
    "num_layers",
    "meeting_layers",
    "num_heads",
    "ffn_dim",
    "dropout",
    "init_std",
    "num_speakers",
    "num_zones",
    "num_reasons",
    "num_span_tags",
)


@keras.saving.register_keras_serializable(package="dialogue_models")
class DependsOnPointer(keras.layers.Layer):
    """Pointer logits from each sentence to every earlier real sentence, plus a "none" column.

    ``logits[b, i, j] = q_i . k_j / sqrt(d)`` for ``j < i`` and sentence ``j`` real, else a
    large negative value; ``logits[b, i, S]`` is a learned "depends on nothing" score.
    """

    def __init__(self, d_model: int, init_std: float = 0.02, **kwargs: Any) -> None:
        super().__init__(**kwargs)
        self.d_model = d_model
        self.init_std = init_std
        init = keras.initializers.TruncatedNormal(stddev=init_std)
        self.query = keras.layers.Dense(d_model, kernel_initializer=init, name="query")
        self.key = keras.layers.Dense(d_model, kernel_initializer=init, name="key")
        self.none = keras.layers.Dense(1, kernel_initializer=init, name="none")

    def build(self, input_shape: tuple[int | None, ...]) -> None:
        for layer in (self.query, self.key, self.none):
            layer.build(input_shape)
        super().build(input_shape)

    def call(self, hidden, sentence_mask):
        h = ops.cast(hidden, "float32")
        scores = ops.matmul(self.query(h), ops.swapaxes(self.key(h), 1, 2))
        scores = scores / ops.sqrt(ops.cast(self.d_model, "float32"))
        s = ops.shape(h)[1]
        earlier = ops.arange(s)[None, :] < ops.arange(s)[:, None]  # (S, S): j < i
        allowed = ops.logical_and(earlier[None, :, :], sentence_mask[:, None, :])
        scores = ops.where(allowed, scores, NEG)
        return ops.concatenate([scores, self.none(h)], axis=-1)

    def compute_output_shape(self, hidden_shape, sentence_mask_shape=None) -> tuple:
        s = hidden_shape[1]
        return (hidden_shape[0], s, None if s is None else s + 1)

    def get_config(self) -> dict[str, Any]:
        return {**super().get_config(), "d_model": self.d_model, "init_std": self.init_std}


@keras.saving.register_keras_serializable(package="dialogue_models")
class NotesModel(keras.Model):
    """Sentence encoder + meeting encoder + heads (see module docstring)."""

    def __init__(
        self,
        vocab_size: int,
        max_len: int,
        d_model: int,
        num_layers: int,
        meeting_layers: int,
        num_heads: int,
        ffn_dim: int,
        max_sentences: int,
        dropout: float = 0.1,
        init_std: float = 0.02,
        num_speakers: int = 8,
        num_zones: int = 5,
        num_reasons: int = 9,
        num_span_tags: int = 5,
        **kwargs: Any,
    ) -> None:
        super().__init__(**kwargs)
        self.args = {
            "vocab_size": vocab_size,
            "max_len": max_len,
            "d_model": d_model,
            "num_layers": num_layers,
            "meeting_layers": meeting_layers,
            "num_heads": num_heads,
            "ffn_dim": ffn_dim,
            "max_sentences": max_sentences,
            "dropout": dropout,
            "init_std": init_std,
            "num_speakers": num_speakers,
            "num_zones": num_zones,
            "num_reasons": num_reasons,
            "num_span_tags": num_span_tags,
        }
        init = keras.initializers.TruncatedNormal(stddev=init_std)
        self.encoder = TransformerEncoder(
            vocab_size,
            max_len,
            d_model,
            num_layers,
            num_heads,
            ffn_dim,
            dropout,
            init_std,
            num_speakers,
            name="encoder",
        )
        self.sentence_position = keras.layers.Embedding(
            max_sentences, d_model, embeddings_initializer=init, name="sentence_position"
        )
        self.meeting_speaker = keras.layers.Embedding(
            num_speakers + 1, d_model, embeddings_initializer=init, name="meeting_speaker"
        )
        self.speaker_change = keras.layers.Embedding(
            2, d_model, embeddings_initializer=init, name="speaker_change"
        )
        self.meeting_dropout = keras.layers.Dropout(dropout)
        self.meeting_blocks = [
            TransformerEncoderBlock(
                d_model, num_heads, ffn_dim, dropout, init_std, name=f"meeting_block_{i}"
            )
            for i in range(meeting_layers)
        ]
        self.meeting_norm = keras.layers.LayerNormalization(epsilon=1e-6, name="meeting_norm")

        def head(units: int, name: str) -> keras.layers.Dense:
            return keras.layers.Dense(units, kernel_initializer=init, dtype="float32", name=name)

        self.importance_head = head(1, "importance_head")
        self.zone_head = head(num_zones, "zone_head")
        self.reason_head = head(num_reasons, "reason_head")
        self.span_head = head(num_span_tags, "span_head")
        self.pointer = DependsOnPointer(d_model, init_std, dtype="float32", name="depends_on_head")

    def call(self, inputs, training: bool | None = None):
        token_ids = ops.cast(inputs["token_ids"], "int32")
        speakers = ops.cast(inputs["speaker_ids"], "int32")
        b, s, t = ops.shape(token_ids)
        real_tokens = ops.not_equal(token_ids, PAD_ID)
        token_speakers = ops.where(real_tokens, speakers[:, :, None], 0)
        hidden = self.encoder(
            ops.reshape(token_ids, (-1, t)), ops.reshape(token_speakers, (-1, t)), training=training
        )
        d = self.args["d_model"]
        tokens = ops.reshape(hidden, (b, s, t, d))
        sentence_mask = real_tokens[:, :, 0]

        previous = ops.concatenate([speakers[:, :1], speakers[:, :-1]], axis=1)
        first = ops.arange(s)[None, :] == 0
        change = ops.cast(ops.logical_or(first, ops.not_equal(speakers, previous)), "int32")
        x = tokens[:, :, 0, :]
        x = x + ops.cast(self.sentence_position(ops.arange(s)), x.dtype)[None, :, :]
        x = x + ops.cast(self.meeting_speaker(speakers), x.dtype)
        x = x + ops.cast(self.speaker_change(change), x.dtype)
        x = self.meeting_dropout(x, training=training)
        for block in self.meeting_blocks:
            x = block(x, padding_mask=sentence_mask, training=training)
        x = self.meeting_norm(x)

        return {
            "importance": ops.squeeze(self.importance_head(x), axis=-1),
            "zone": self.zone_head(x),
            "reason": self.reason_head(x),
            "depends_on": self.pointer(x, sentence_mask),
            "spans": self.span_head(tokens),
        }

    def get_config(self) -> dict[str, Any]:
        return {**super().get_config(), **self.args}


def build_notes_model(
    model_cfg: dict[str, Any], max_sentences: int, sentence_tokens: int
) -> NotesModel:
    """Build (and run once, so every weight exists) a student from the ``model`` config."""
    model = NotesModel(**{k: model_cfg[k] for k in MODEL_KEYS}, max_sentences=max_sentences)
    model(
        {
            "token_ids": ops.ones((1, 2, sentence_tokens), dtype="int32"),
            "speaker_ids": ops.ones((1, 2), dtype="int32"),
        }
    )
    return model


def load_sentence_encoder(model: NotesModel, checkpoint: str | Path) -> None:
    """Copy the MLM ``encoder.keras`` weights into the sentence encoder (same architecture)."""
    load_encoder_weights(model, checkpoint)


def masked_sparse_ce() -> keras.losses.Loss:
    """Sparse categorical cross-entropy on logits, averaged over the sample weights."""
    return keras.losses.SparseCategoricalCrossentropy(
        from_logits=True, reduction="mean_with_sample_weight"
    )


def label_weights(labels: np.ndarray, class_weights: Sequence[float] | None = None) -> np.ndarray:
    """Per-position float32 weights: 0 where ``labels == IGNORE``, else the class weight (or 1)."""
    valid = labels != IGNORE
    if class_weights is None:
        return valid.astype(np.float32)
    table = np.asarray(class_weights, dtype=np.float32)
    return np.where(valid, table[np.where(valid, labels, 0)], 0.0).astype(np.float32)


def safe_labels(labels: np.ndarray) -> np.ndarray:
    """Replace ``IGNORE`` by 0 so the loss can index; pair with :func:`label_weights`."""
    return np.where(labels == IGNORE, 0, labels).astype(np.int32)


def inverse_sqrt_frequency_weights(counts: Sequence[int]) -> list[float]:
    """Class weights proportional to ``1 / sqrt(frequency)``, scaled so the average weight over
    labelled positions is 1 (``sum_c f_c * w_c = 1``). Classes with zero count get weight 0.
    """
    c = np.asarray(counts, dtype=np.float64)
    freq = c / c.sum()
    w = np.zeros_like(freq)
    nz = freq > 0
    w[nz] = 1.0 / np.sqrt(freq[nz])
    w /= (freq * w).sum()
    return [round(float(x), 6) for x in w]


#: ``y_true[..., k]`` of :class:`NotesImportanceLoss`.
IMPORTANCE_CHANNELS = (
    "gold",
    "gold_weight",
    "qwen_p",
    "qwen_weight",
    "roberta_logit",
    "roberta_weight",
)


#: Zones that AMI never uses (indices in ``ZONES``: context, closing).
AMI_MASKED_ZONES = (0, 4)


_EPS = 1e-4


_NEG = -1e9


def _binary_kl(teacher_logit, student_logit, temperature: float):
    """``KL(Bern(sigmoid(t / T)) || Bern(sigmoid(s / T))) * T**2``."""
    p = ops.sigmoid(teacher_logit / temperature)
    log_q = ops.log_sigmoid(student_logit / temperature)
    log_1q = ops.log_sigmoid(-student_logit / temperature)
    p = ops.clip(p, _EPS, 1.0 - _EPS)
    kl = p * (ops.log(p) - log_q) + (1.0 - p) * (ops.log(1.0 - p) - log_1q)
    return kl * temperature**2


@keras.saving.register_keras_serializable(package="dialogue_models")
class NotesImportanceLoss(keras.losses.Loss):
    """``w_gold * conf * BCE(gold) + w_qwen * KL(Qwen) + w_roberta * KL(RoBERTa)`` per sentence."""

    def __init__(
        self,
        gold: float = 1.0,
        qwen: float = 0.0,
        roberta: float = 0.0,
        temperature: float = 2.0,
        name: str = "notes_importance_loss",
        reduction: str = "mean_with_sample_weight",
        **kwargs: Any,
    ) -> None:
        super().__init__(name=name, reduction=reduction, **kwargs)
        self.gold, self.qwen, self.roberta, self.temperature = gold, qwen, roberta, temperature

    def call(self, y_true, y_pred):
        y = ops.cast(y_true, "float32")
        logit = ops.cast(y_pred, "float32")
        gold, gold_w, qwen_p, qwen_w, rob_logit, rob_w = (y[..., k] for k in range(6))
        target = ops.clip(gold, 0.0, 1.0)
        loss = self.gold * gold_w * ops.binary_crossentropy(target, logit, from_logits=True)
        if self.qwen:
            p = ops.clip(qwen_p, _EPS, 1.0 - _EPS)
            qwen_logit = ops.log(p) - ops.log(1.0 - p)
            loss += self.qwen * qwen_w * _binary_kl(qwen_logit, logit, self.temperature)
        if self.roberta:
            loss += self.roberta * rob_w * _binary_kl(rob_logit, logit, self.temperature)
        return loss

    def get_config(self) -> dict[str, Any]:
        return {
            **super().get_config(),
            "gold": self.gold,
            "qwen": self.qwen,
            "roberta": self.roberta,
            "temperature": self.temperature,
        }


@keras.saving.register_keras_serializable(package="dialogue_models")
class ZoneLoss(keras.losses.Loss):
    """Sparse CE on zone logits; ``y_true[..., 0]`` = zone, ``y_true[..., 1]`` = 1 for AMI."""

    def __init__(
        self,
        name: str = "zone_loss",
        reduction: str = "mean_with_sample_weight",
        **kwargs: Any,
    ) -> None:
        super().__init__(name=name, reduction=reduction, **kwargs)

    def call(self, y_true, y_pred):
        y = ops.cast(y_true, "int32")
        logits = ops.cast(y_pred, "float32")
        n = ops.shape(logits)[-1]
        masked = ops.zeros((n,), dtype="bool")
        for k in AMI_MASKED_ZONES:
            masked = ops.logical_or(masked, ops.arange(n) == k)
        is_ami = ops.equal(y[..., 1], 1)[..., None]
        logits = ops.where(ops.logical_and(is_ami, masked), _NEG, logits)
        labels = ops.clip(y[..., 0], 0, n - 1)
        return ops.sparse_categorical_crossentropy(labels, logits, from_logits=True)

In [ ]:
from __future__ import annotations


import csv
from datetime import UTC, datetime


def set_seed(seed: int) -> None:
    """Seed Python's ``random``, NumPy and Keras (which also seeds the backend)."""
    random.seed(seed)
    np.random.seed(seed)  # noqa: NPY002  (global state is what we seed)
    keras.utils.set_random_seed(seed)


NO_DECAY = ["bias", "gamma", "beta", "position_embedding"]


@keras.saving.register_keras_serializable(package="dialogue_models")
class LayerwiseAdamW(keras.optimizers.AdamW):
    """AdamW whose learning rate (and decoupled weight decay) is scaled per layer.

    ``lr_multipliers`` maps a layer name to a factor: a variable whose path contains that
    name as one segment (``.../encoder/...``) uses ``learning_rate * factor``. Other variables
    use the base learning rate.
    """

    def __init__(self, lr_multipliers: dict[str, float] | None = None, **kwargs: Any) -> None:
        super().__init__(**kwargs)
        self.lr_multipliers = dict(lr_multipliers or {})

    def _multiplier(self, variable) -> float:
        segments = variable.path.split("/")
        for name, factor in self.lr_multipliers.items():
            if name in segments:
                return factor
        return 1.0

    def update_step(self, gradient, variable, learning_rate):
        return super().update_step(gradient, variable, learning_rate * self._multiplier(variable))

    def _apply_weight_decay(self, variables) -> None:
        if self.weight_decay is None:
            return
        for variable in variables:
            if self._use_weight_decay(variable):
                lr = keras.ops.cast(self.learning_rate, variable.dtype) * self._multiplier(variable)
                wd = keras.ops.cast(self.weight_decay, variable.dtype)
                variable.assign(variable - variable * wd * lr)

    def get_config(self) -> dict[str, Any]:
        return {**super().get_config(), "lr_multipliers": self.lr_multipliers}


def build_optimizer(opt_cfg: dict[str, Any], total_steps: int) -> keras.optimizers.Optimizer:
    """AdamW with linear warmup to the peak learning rate, then cosine decay to 0.

    The peak is ``peak_lr`` for every weight, or ``lr_heads`` with ``lr_encoder`` for the
    weights of the layer named ``encoder`` (students: smaller steps for pretrained weights).
    ``warmup_steps`` (absolute) or ``warmup_fraction`` (of ``total_steps``) sets the warmup.
    """
    if opt_cfg["name"] != "adamw" or opt_cfg["schedule"] != "cosine":
        raise ValueError("Only adamw with a cosine schedule is implemented.")
    if "warmup_steps" in opt_cfg:
        warmup = int(opt_cfg["warmup_steps"])
    else:
        warmup = int(opt_cfg["warmup_fraction"] * total_steps)
    warmup = min(warmup, max(total_steps - 1, 0))
    if "lr_heads" in opt_cfg:
        peak = opt_cfg["lr_heads"]
        multipliers = {"encoder": opt_cfg["lr_encoder"] / peak}
    else:
        peak, multipliers = opt_cfg["peak_lr"], {}
    schedule = keras.optimizers.schedules.CosineDecay(
        initial_learning_rate=0.0,
        decay_steps=max(total_steps - warmup, 1),
        warmup_target=peak,
        warmup_steps=warmup,
    )
    optimizer = LayerwiseAdamW(
        lr_multipliers=multipliers, learning_rate=schedule, weight_decay=opt_cfg["weight_decay"]
    )
    optimizer.exclude_from_weight_decay(var_names=NO_DECAY)
    return optimizer


class TimeGuard(keras.callbacks.Callback):
    """Stop training cleanly before ``max_hours`` after ``session_start`` (default: train start).

    Pass the same ``session_start`` to every run of a session (several seeds in one Kaggle
    session share one budget). At each epoch end, stops if another epoch (estimated as 1.2 x
    the longest so far) would overrun. Also stops mid-epoch if the hard limit is reached
    (checked on every batch-end call: with ``steps_per_execution`` > 1 Keras calls it once per
    execution, with ``batch`` = the last step of that execution).
    ``stopped`` tells the caller whether training was cut short (so it can skip the final
    evaluation and resume later).
    """

    def __init__(self, max_hours: float, session_start: float | None = None) -> None:
        super().__init__()
        self.limit = max_hours * 3600.0
        self.session_start = session_start
        self.stopped = False
        self._start = 0.0
        self._epoch_start = 0.0
        self._longest_epoch = 0.0

    def on_train_begin(self, logs=None) -> None:
        self._start = self.session_start if self.session_start is not None else time.time()

    def on_epoch_begin(self, epoch, logs=None) -> None:
        self._epoch_start = time.time()

    def on_train_batch_end(self, batch, logs=None) -> None:
        if time.time() - self._start > self.limit:
            log.warning("TimeGuard: hard limit reached mid-epoch, stopping.")
            self.stopped = self.model.stop_training = True

    def on_epoch_end(self, epoch, logs=None) -> None:
        now = time.time()
        self._longest_epoch = max(self._longest_epoch, now - self._epoch_start)
        if epoch + 1 >= (self.params or {}).get("epochs", math.inf):
            return  # the last epoch: training ends anyway, nothing was cut short
        if now - self._start + 1.2 * self._longest_epoch > self.limit:
            log.warning("TimeGuard: next epoch would exceed %.2f h, stopping.", self.limit / 3600)
            self.stopped = self.model.stop_training = True


class SyncEpoch(keras.callbacks.Callback):
    """Call ``dataset.set_epoch(epoch)`` when each epoch starts.

    A dataset that counted epochs itself would restart at 0 after ``BackupAndRestore``
    resumes a run (and replay the first epoch's shuffle); Keras passes the true epoch here.
    """

    def __init__(self, dataset: Any) -> None:
        super().__init__()
        self.dataset = dataset

    def on_epoch_begin(self, epoch, logs=None) -> None:
        self.dataset.set_epoch(epoch)


def pad_rows(rows: np.ndarray, batch_size: int) -> tuple[np.ndarray, int]:
    """Pad a partial batch of row indices to ``batch_size`` by repeating the last row.

    Returns ``(rows, n_real)``; give rows ``n_real:`` zero weight (or drop their outputs).
    Data parallelism splits every batch evenly across devices, so batches are always full.
    """
    rows = np.asarray(rows)
    n_real = len(rows)
    if 0 < n_real < batch_size:
        rows = np.concatenate([rows, np.repeat(rows[-1:], batch_size - n_real)])
    return rows, n_real


def configure_devices(
    mixed_precision: str, *batch_sizes: int, data_parallel: bool = True
) -> dict[str, Any]:
    """Set the dtype policy and, with several GPUs and ``data_parallel``, data parallelism.
    Returns device info.

    With data parallelism every ``batch_sizes`` value must divide evenly across the devices.
    ``data_parallel=False`` uses one GPU: no cross-GPU communication (NCCL failed on the 2x T4
    image of 2026-10, ``Cuda failure 500 'named symbol not found'``).
    """
    keras.mixed_precision.set_dtype_policy(mixed_precision)
    devices = jax.devices() if data_parallel else jax.devices()[:1]
    for bs in batch_sizes:
        if bs % len(devices):
            raise ValueError(f"batch size {bs} is not divisible by the {len(devices)} devices")
    if len(devices) > 1:
        keras.distribution.set_distribution(keras.distribution.DataParallel())
    return {
        "devices": [str(d) for d in devices],
        "platform": devices[0].platform,
        "mixed_precision": mixed_precision,
        "keras": keras.__version__,
        "jax": jax.__version__,
    }


def best_logged_value(csv_path: Path, key: str, mode: str) -> float | None:
    """Best value of ``key`` in a CSVLogger file (for resuming checkpoint thresholds)."""
    if not csv_path.exists():
        return None
    with csv_path.open() as f:
        values = [float(row[key]) for row in csv.DictReader(f) if row.get(key)]
    if not values:
        return None
    return max(values) if mode == "max" else min(values)


def write_json(path: Path, data: dict[str, Any]) -> None:
    """Write ``data`` as indented JSON, creating parent directories."""
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(data, indent=2, default=str) + "\n")


def utc_now() -> str:
    """Current UTC time, ISO format, seconds precision."""
    return datetime.now(UTC).isoformat(timespec="seconds")


def f1_score(truth: np.ndarray, pred: np.ndarray) -> float:
    """Binary F1 (0 when there is no positive prediction or label)."""
    truth, pred = np.asarray(truth, dtype=bool), np.asarray(pred, dtype=bool)
    tp = float(np.sum(truth & pred))
    denom = float(np.sum(truth) + np.sum(pred))
    return 0.0 if denom == 0 else 2 * tp / denom


def tune_threshold(truth: np.ndarray, prob: np.ndarray) -> tuple[float, float]:
    """Threshold in 0.05..0.95 that maximises importance F1, and that F1."""
    truth = np.asarray(truth).astype(bool)
    grid = np.round(np.arange(0.05, 0.96, 0.05), 2)
    scores = [f1_score(truth, np.asarray(prob) >= t) for t in grid]
    best = int(np.argmax(scores))
    return float(grid[best]), float(scores[best])


def average_precision(truth: np.ndarray, prob: np.ndarray) -> float:
    """Area under the precision-recall curve (step-wise, as in scikit-learn)."""
    truth = np.asarray(truth, dtype=bool)
    if not truth.any():
        return 0.0
    order = np.argsort(-np.asarray(prob, dtype=np.float64), kind="stable")
    hits = truth[order]
    precision = np.cumsum(hits) / np.arange(1, len(hits) + 1)
    return float(np.sum(precision[hits]) / truth.sum())


def zone_macro_f1(truth: np.ndarray, pred: np.ndarray) -> float:
    """Macro F1 over the zones present in ``truth``."""
    truth, pred = np.asarray(truth), np.asarray(pred)
    present = np.unique(truth)
    if not len(present):
        return 0.0
    return float(np.mean([f1_score(truth == z, pred == z) for z in present]))


def span_entities(tags: Sequence[int]) -> set[tuple[int, int, str]]:
    """``(start, end, role)`` entities of one BIO sequence (an ``I`` without ``B`` starts one)."""
    entities, start, role = set(), None, None
    for i, t in enumerate([*tags, 0]):
        name = SPAN_TAGS[t] if 0 <= t < len(SPAN_TAGS) else "O"
        inside = name.startswith("I-") and role == name[2:]
        if start is not None and not inside:
            entities.add((start, i, role))
            start, role = None, None
        if name != "O" and not inside:
            start, role = i, name[2:]
    return entities


def span_f1(gold: np.ndarray, pred: np.ndarray) -> dict[str, float]:
    """Exact-match entity F1 per role and overall; ``gold`` / ``pred`` are (N, T) BIO tags
    (``-1`` positions are ignored, i.e. treated as ``O``)."""
    counts = {"owner": [0, 0, 0], "deadline": [0, 0, 0]}  # tp, n_gold, n_pred
    for g_row, p_row in zip(np.asarray(gold), np.asarray(pred), strict=True):
        valid = g_row >= 0
        g = span_entities(np.where(valid, g_row, 0).tolist())
        p = span_entities(np.where(valid, p_row, 0).tolist())
        for role, c in counts.items():
            gr, pr = {e for e in g if e[2] == role}, {e for e in p if e[2] == role}
            c[0] += len(gr & pr)
            c[1] += len(gr)
            c[2] += len(pr)

    def f1(tp: int, n_gold: int, n_pred: int) -> float:
        return 0.0 if n_gold + n_pred == 0 else 2 * tp / (n_gold + n_pred)

    total = [sum(c[k] for c in counts.values()) for k in range(3)]
    return {
        "owner_f1": round(f1(*counts["owner"]), 4),
        "deadline_f1": round(f1(*counts["deadline"]), 4),
        "span_f1": round(f1(*total), 4),
        "gold_spans": total[1],
    }


def selection_metrics(
    truth: np.ndarray,
    zone_true: np.ndarray,
    picked: np.ndarray,
    prob: np.ndarray | None = None,
    zone_pred: np.ndarray | None = None,
    hard_negative: np.ndarray | None = None,
) -> dict[str, Any]:
    """Metrics of one system on one set of sentences (see module docstring); ``None`` for
    what the system does not predict."""
    truth, picked = np.asarray(truth, dtype=bool), np.asarray(picked, dtype=bool)
    zone_true = np.asarray(zone_true)
    out: dict[str, Any] = {
        "sentences": int(len(truth)),
        "importance_f1": round(f1_score(truth, picked), 4),
        "pr_auc": None if prob is None else round(average_precision(truth, prob), 4),
        "zone_macro_f1": None,
        "decision_f1": None,
        "action_f1": None,
        "hard_negative_fp": None,
    }
    if zone_pred is not None:
        zone_pred = np.asarray(zone_pred)
        out["zone_macro_f1"] = round(zone_macro_f1(zone_true, zone_pred), 4)
        for key, z in (("decision_f1", DECISION), ("action_f1", ACTION)):
            out[key] = round(f1_score(truth & (zone_true == z), picked & (zone_pred == z)), 4)
    if hard_negative is not None:
        hn = np.asarray(hard_negative, dtype=bool) & ~truth
        out["hard_negative_fp"] = round(float(picked[hn].mean()), 4) if hn.any() else None
    return out


AMI = SOURCES.index("ami")


def empty_teacher(n_rows: int) -> dict[str, np.ndarray]:
    """Per-row teacher arrays with no labels."""
    return {
        "qwen_score": np.full(n_rows, -1.0, dtype=np.float32),
        "qwen_reason": np.full(n_rows, IGNORE, dtype=np.int8),
        "qwen_depends": np.full(n_rows, -2, dtype=np.int64),  # row; -1 = none, -2 = unlabelled
        "roberta_logit": np.full(n_rows, np.nan, dtype=np.float32),
    }


def align_teacher(
    dataset: NotesSet, teacher_dir: str | Path | None, split: str, teachers: list[str]
) -> dict[str, np.ndarray]:
    """Teacher targets for every sentence row of ``dataset`` (see module docstring)."""
    arrays = empty_teacher(len(dataset["importance"]))
    if teacher_dir is None:
        return arrays
    rows = sentence_rows(dataset)
    path = Path(teacher_dir) / "qwen" / f"{split}.parquet"
    if "qwen" in teachers and path.exists():
        for r in pq.read_table(path).to_pylist():
            row = rows.get((r["note_id"], int(r["sentence_id"])))
            if row is None:
                continue
            arrays["qwen_score"][row] = r["score"]
            reason = r["reason"]
            arrays["qwen_reason"][row] = REASONS.index(reason) if reason in REASONS else IGNORE
            dep = int(r["depends_on"])
            arrays["qwen_depends"][row] = -1 if dep < 0 else rows.get((r["note_id"], dep), -1)
    path = Path(teacher_dir) / "roberta" / f"{split}.parquet"
    if "roberta" in teachers and path.exists():
        for r in pq.read_table(path).to_pylist():
            row = rows.get((r["note_id"], int(r["sentence_id"])))
            if row is not None:
                arrays["roberta_logit"][row] = r["logit"]
    return arrays


def bucket_batch_sizes(buckets: list[int], sentences_per_batch: int) -> dict[int, int]:
    """Windows per batch for each bucket: a fixed sentence budget (``sentences_per_batch``)
    keeps the memory of a step about the same whatever the window length."""
    sizes = {b: max(1, sentences_per_batch // b) for b in buckets}
    if sentences_per_batch % max(buckets):
        raise ValueError(f"sentences_per_batch must be a multiple of {max(buckets)}")
    return sizes


def window_batches(
    dataset: NotesSet,
    windows: np.ndarray,
    buckets: list[int],
    sentences_per_batch: int,
    rng: np.random.Generator | None = None,
) -> list[tuple[np.ndarray, int]]:
    """``(window indices, bucket)`` batches; windows of a batch share the smallest bucket and a
    batch holds ``sentences_per_batch // bucket`` windows.

    With ``rng`` (training) windows are shuffled within each bucket and the batch order is
    shuffled; without, the order is deterministic. The last batch of a bucket may be partial.
    """
    lengths = np.asarray(dataset["window_length"])
    sizes = bucket_batch_sizes(buckets, sentences_per_batch)
    by_bucket: dict[int, list[int]] = {}
    for w in windows:
        by_bucket.setdefault(bucket_for(int(lengths[w]), buckets), []).append(int(w))
    batches = []
    for bucket in sorted(by_bucket):
        ws, bs = np.asarray(by_bucket[bucket]), sizes[bucket]
        if rng is not None:
            ws = ws[rng.permutation(len(ws))]
        batches += [(ws[i : i + bs], bucket) for i in range(0, len(ws), bs)]
    if rng is not None:
        batches = [batches[i] for i in rng.permutation(len(batches))]
    return batches


def depends_targets(rows: np.ndarray, depends_row: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """Pointer labels for a batch: position of the dependency in the window, or ``S`` = none.

    ``rows`` (B, S) are the sentence rows of each slot (``-1`` = padding). A dependency outside
    the window (or not earlier) becomes "none"; unlabelled sentences get weight 0.
    """
    b, s = rows.shape
    labels = np.full((b, s), s, dtype=np.int32)
    weights = np.zeros((b, s), dtype=np.float32)
    for i in range(b):
        position = {int(r): j for j, r in enumerate(rows[i]) if r >= 0}
        for j, r in enumerate(rows[i]):
            if r < 0 or depends_row[r] == -2:
                continue
            target = position.get(int(depends_row[r]), s)
            labels[i, j] = target if target < j else s
            weights[i, j] = 1.0
    return labels, weights


class NotesDataset(keras.utils.PyDataset):
    """Batches ``(inputs, targets, sample_weights)`` of meeting windows (see module docstring)."""

    def __init__(
        self,
        dataset: NotesSet,
        teacher: dict[str, np.ndarray],
        windows: np.ndarray,
        cfg: dict[str, Any],
        zone_weights: list[float] | None,
        shuffle: bool,
        seed: int = 0,
        max_batches: int | None = None,
        **kwargs: Any,
    ) -> None:
        super().__init__(**kwargs)
        self.dataset, self.teacher, self.windows = dataset, teacher, np.asarray(windows)
        self.buckets = cfg["data"]["sentence_buckets"]
        self.sentences_per_batch = cfg["training"]["sentences_per_batch"]
        self.zone_weights, self.shuffle, self.seed = zone_weights, shuffle, seed
        self.epoch = 0
        self.max_batches = max_batches
        self._batches = self._make_batches()

    def _make_batches(self) -> list[tuple[np.ndarray, int]]:
        rng = np.random.default_rng([self.seed, self.epoch]) if self.shuffle else None
        batches = window_batches(
            self.dataset, self.windows, self.buckets, self.sentences_per_batch, rng
        )
        return batches if self.max_batches is None else batches[: self.max_batches]

    def set_epoch(self, epoch: int) -> None:
        if epoch != self.epoch:
            self.epoch = epoch
            self._batches = self._make_batches()

    def __len__(self) -> int:
        return len(self._batches)

    def __getitem__(self, index: int):
        windows, bucket = self._batches[index]
        windows, n_real = pad_rows(windows, max(1, self.sentences_per_batch // bucket))
        return batch_arrays(self.dataset, self.teacher, windows, bucket, self.zone_weights, n_real)


def batch_arrays(
    dataset: NotesSet,
    teacher: dict[str, np.ndarray],
    windows: np.ndarray,
    slots: int,
    zone_weights: list[float] | None = None,
    n_real: int | None = None,
) -> tuple[dict[str, np.ndarray], dict[str, np.ndarray], dict[str, np.ndarray]]:
    """Model inputs, targets and sample weights for one batch of windows."""
    g = gather_windows(dataset, windows, slots)
    mask, rows = g["sentence_mask"], g["rows"]
    safe = np.where(mask, rows, 0)
    is_ami = np.broadcast_to((g["source"] == AMI)[:, None], mask.shape)

    def teacher_values(name: str, fill: float) -> np.ndarray:
        return np.where(mask, teacher[name][safe], fill)

    qwen_p = teacher_values("qwen_score", -1.0)
    rob = teacher_values("roberta_logit", np.nan)
    importance = np.stack(
        [
            np.where(mask, g["importance"], 0),
            np.where(mask, g["confidence"], 0.0),
            np.clip(qwen_p, 0.0, 1.0),
            (qwen_p >= 0).astype(np.float32),
            np.nan_to_num(rob),
            (~np.isnan(rob)).astype(np.float32),
        ],
        axis=-1,
    ).astype(np.float32)
    reason = teacher_values("qwen_reason", IGNORE).astype(np.int32)
    depends, depends_w = depends_targets(rows, teacher["qwen_depends"])
    x = {"token_ids": g["token_ids"], "speaker_ids": g["speaker"].astype(np.int32)}
    y = {
        "importance": importance,
        "zone": np.stack([safe_labels(g["zone"]), is_ami.astype(np.int32)], axis=-1),
        "reason": safe_labels(reason),
        "depends_on": depends,
        "spans": safe_labels(g["span_tags"]),
    }
    sw = {
        "importance": mask.astype(np.float32),
        "zone": label_weights(g["zone"], zone_weights),
        "reason": label_weights(reason),
        "depends_on": depends_w,
        "spans": label_weights(g["span_tags"]),
    }
    if n_real is not None:
        for weights in sw.values():
            weights[n_real:] = 0.0  # repeated windows of a partial batch do not count
    return x, y, sw


def train_windows(dataset: NotesSet, oversample: int) -> np.ndarray:
    """All windows, with windows of hard-negative meetings repeated ``oversample`` times."""
    windows = np.arange(dataset.num_windows)
    hn = np.asarray(dataset["source"])[np.asarray(dataset["window_note"])] == 1
    return np.concatenate([windows, *[windows[hn]] * (oversample - 1)])


def zone_class_weights(cfg: dict[str, Any], dataset: NotesSet) -> list[float] | None:
    """Inverse-sqrt-frequency zone weights from the train labels, or ``None``."""
    mode = cfg["loss"].get("zone_class_weights")
    if mode is None:
        return None
    if mode != "inverse_sqrt":
        raise ValueError(f"Unknown zone_class_weights: {mode!r}")
    counts = np.bincount(np.asarray(dataset["zone"]), minlength=len(ZONES))
    return inverse_sqrt_frequency_weights(counts.tolist())


def compile_model(cfg: dict[str, Any], total_steps: int) -> keras.Model:
    """Student (MLM or random sentence encoder) compiled with the configured losses."""
    data, loss_cfg = cfg["data"], cfg["loss"]
    model = build_notes_model(cfg["model"], data["max_sentences"], data["max_sentence_tokens"])
    checkpoint = cfg["paths"].get("init_checkpoint")
    if checkpoint:
        load_sentence_encoder(model, checkpoint)
    losses = {
        "importance": NotesImportanceLoss(
            loss_cfg["importance"],
            loss_cfg["qwen_kl"],
            loss_cfg["roberta_kl"],
            loss_cfg["temperature"],
        ),
        "zone": ZoneLoss(),
        "spans": masked_sparse_ce(),
        "reason": masked_sparse_ce(),
        "depends_on": masked_sparse_ce(),
    }
    weights = {
        "importance": 1.0,
        "zone": loss_cfg["zone"],
        "spans": loss_cfg["spans"],
        "reason": loss_cfg["reason"],
        "depends_on": loss_cfg["depends_on"],
    }
    losses = {k: v for k, v in losses.items() if weights[k]}
    opt = cfg["optimizer"]
    optimizer = build_optimizer(
        {
            "name": opt["name"],
            "schedule": opt["schedule"],
            "lr_heads": opt["lr_meeting"],
            "lr_encoder": opt["lr_sentence_encoder"],
            "warmup_fraction": opt["warmup_fraction"],
            "weight_decay": opt["weight_decay"],
        },
        total_steps,
    )
    model.compile(optimizer=optimizer, loss=losses, loss_weights={k: weights[k] for k in losses})
    return model


def predict_sentences(
    model: keras.Model, dataset: NotesSet, cfg: dict[str, Any]
) -> dict[str, np.ndarray]:
    """Per sentence row: importance probability, zone probabilities and span tags per token.

    Sentences covered by several windows get the mean of their importance / zone predictions
    (span tags: the last window's).
    """
    n = len(dataset["importance"])
    prob, zones, seen = np.zeros(n), np.zeros((n, len(ZONES))), np.zeros(n)
    spans = np.zeros(np.asarray(dataset["span_tags"]).shape, dtype=np.int8)
    per_batch = cfg["training"]["sentences_per_batch"]
    teacher = empty_teacher(n)
    for windows, bucket in window_batches(
        dataset, np.arange(dataset.num_windows), cfg["data"]["sentence_buckets"], per_batch
    ):
        padded, n_real = pad_rows(windows, max(1, per_batch // bucket))
        x, _, _ = batch_arrays(dataset, teacher, padded, bucket)
        rows = gather_windows(dataset, padded[:n_real], bucket)["rows"]
        out = model.predict_on_batch(x)
        p = 1.0 / (1.0 + np.exp(-np.asarray(out["importance"], dtype=np.float64)))
        z = np.asarray(out["zone"], dtype=np.float64)
        z = np.exp(z - z.max(-1, keepdims=True))
        z /= z.sum(-1, keepdims=True)
        valid = rows >= 0
        np.add.at(prob, rows[valid], p[:n_real][valid])
        np.add.at(zones, rows[valid], z[:n_real][valid])
        np.add.at(seen, rows[valid], 1.0)
        spans[rows[valid]] = np.argmax(np.asarray(out["spans"]), axis=-1)[:n_real][valid]
    seen = np.maximum(seen, 1.0)
    return {"importance": prob / seen, "zone": zones / seen[:, None], "spans": spans}


CALIBRATED = ("decision", "action_item")


def tune_zone_thresholds(
    importance: np.ndarray,
    zone: np.ndarray,
    kept: np.ndarray,
    zone_prob: np.ndarray,
    grid: Sequence[float],
    groups: Sequence[np.ndarray] | None = None,
) -> dict[str, float]:
    """Threshold per calibrated zone maximising F1 of ``kept & p(z) >= t`` against the gold
    ``importance & zone == z`` (all arrays per sentence; ``zone_prob`` is (N, 5)).

    With ``groups`` (boolean row masks, e.g. AMI and synthetic), the mean of the per-group
    F1 is maximised, so a small group (real meetings) weighs as much as a large one."""
    importance, kept = np.asarray(importance, dtype=bool), np.asarray(kept, dtype=bool)
    masks = list(groups) if groups else [np.ones_like(importance)]
    thresholds = {}
    for name in CALIBRATED:
        z = ZONES.index(name)
        truth = importance & (np.asarray(zone) == z)

        scores = [
            float(np.mean([f1_score(truth[m], (kept & (zone_prob[:, z] >= t))[m]) for m in masks]))
            for t in grid
        ]
        thresholds[name] = float(grid[int(np.argmax(scores))])
    return thresholds


def assign_zones(zone_prob: np.ndarray, thresholds: dict[str, float] | None) -> np.ndarray:
    """Zone index per sentence: argmax, overridden by calibrated zones above their threshold."""
    zones = np.asarray(zone_prob).argmax(-1)
    if not thresholds:
        return zones
    ratios = np.stack([zone_prob[:, ZONES.index(z)] / thresholds[z] for z in CALIBRATED], axis=-1)
    best = ratios.argmax(-1)
    passes = ratios.max(-1) >= 1.0
    calibrated = np.array([ZONES.index(z) for z in CALIBRATED])[best]
    return np.where(passes, calibrated, zones)


def calibrate_zones(
    val_data: NotesSet, val_preds: dict[str, np.ndarray], threshold: float, cfg: dict[str, Any]
) -> dict[str, float] | None:
    """Zone thresholds tuned on validation. ``inference.zone_calibration``:

    - ``ami``: AMI validation only (finds rare real decisions, over-labels clean meetings);
    - ``all``: every validation sentence pooled (synthetic dominates; AMI decisions vanish);
    - ``balanced``: the mean of AMI F1 and synthetic F1, so both kinds of meeting count;
    - ``null``: argmax zones.
    """
    mode = cfg["inference"].get("zone_calibration")
    if mode is None:
        return None
    if mode not in ("ami", "all", "balanced"):
        raise ValueError(f"Unknown zone_calibration: {mode!r}")
    lengths = np.asarray(val_data["note_length"])
    ami = np.repeat(np.asarray(val_data["source"]) == AMI, lengths)
    rows = ami if mode == "ami" else np.ones(len(ami), dtype=bool)
    groups = [ami[rows], ~ami[rows]] if mode == "balanced" else None
    grid = np.round(np.arange(*cfg["inference"]["zone_threshold_grid"]), 3)
    return tune_zone_thresholds(
        np.asarray(val_data["importance"])[rows],
        np.asarray(val_data["zone"])[rows],
        val_preds["importance"][rows] >= threshold,
        val_preds["zone"][rows],
        grid,
        groups,
    )


def sentence_metrics(
    dataset: NotesSet,
    preds: dict[str, np.ndarray],
    threshold: float,
    zone_thresholds: dict[str, float] | None = None,
) -> dict[str, Any]:
    """:func:`~src.model1_notes.metrics.selection_metrics` overall and per source, plus span
    F1 on the sentences with gold spans. Zones use ``zone_thresholds`` when given
    (:mod:`~src.model1_notes.calibrate`), else the argmax."""
    note_of_row = np.repeat(np.arange(len(dataset.note_ids)), np.asarray(dataset["note_length"]))
    source = np.asarray(dataset["source"])[note_of_row]
    truth = np.asarray(dataset["importance"]).astype(bool)
    zone_true = np.asarray(dataset["zone"])
    zone_pred = assign_zones(preds["zone"], zone_thresholds)
    picked = preds["importance"] >= threshold
    hard = np.asarray(dataset["hard_negative"])

    def block(rows: np.ndarray) -> dict[str, Any]:
        return selection_metrics(
            truth[rows],
            zone_true[rows],
            picked[rows],
            preds["importance"][rows],
            zone_pred[rows],
            hard[rows],
        )

    result = {"all": block(np.ones_like(truth))}
    for k, name in enumerate(SOURCES):
        if (source == k).any():
            result[name] = block(source == k)
    if "spans" in preds:
        result["spans"] = span_f1(np.asarray(dataset["span_tags"]), preds["spans"])
    return result


def run_paths(cfg: dict[str, Any], seed: int) -> dict[str, Path]:
    """Checkpoint and results locations for one seed."""
    out = Path(cfg["paths"]["output_dir"]) / f"seed_{seed}"
    results = Path(cfg["paths"]["results_dir"]) / cfg["name"] / f"seed_{seed}"
    return {
        "checkpoints": out,
        "backup": out / "backup",
        "best_model": out / "best.keras",
        "results": results,
        "history": results / "history.csv",
    }


def train_seed(cfg: dict[str, Any], seed: int, session_start: float | None = None) -> dict:
    """Train (or resume) one seed, then write validation metrics and test predictions."""
    paths = run_paths(cfg, seed)
    if (paths["results"] / "metrics.json").exists():
        raise FileExistsError(f"{paths['results']} is complete; use a new experiment name.")
    tr, es = cfg["training"], cfg["training"]["early_stopping"]
    start, started = time.time(), utc_now()
    set_seed(seed)
    sizes = bucket_batch_sizes(cfg["data"]["sentence_buckets"], tr["sentences_per_batch"])
    device_info = configure_devices(
        tr["mixed_precision"], *sizes.values(), data_parallel=tr["data_parallel"]
    )
    processed, teachers = Path(cfg["paths"]["processed_dir"]), cfg.get("teachers", [])
    train_data = NotesSet.load(processed / "train")
    val_data = NotesSet.load(processed / "validation")

    def teacher_for(split: str, data: NotesSet) -> dict[str, np.ndarray]:
        return align_teacher(data, cfg["paths"].get("teacher_dir"), split, teachers)

    zone_w = zone_class_weights(cfg, train_data)
    windows = train_windows(train_data, cfg["data"]["hard_negative_oversample"])
    probe = NotesDataset(train_data, {}, windows, cfg, zone_w, shuffle=False)
    spe, epochs = len(probe), tr["epochs"]
    if tr["max_steps"] is not None:
        spe = min(spe, tr["max_steps"])
        epochs = min(epochs, math.ceil(tr["max_steps"] / spe))
    train_ds = NotesDataset(
        train_data,
        teacher_for("train", train_data),
        windows,
        cfg,
        zone_w,
        shuffle=True,
        seed=seed,
        max_batches=spe,
    )
    val_ds = NotesDataset(
        val_data,
        teacher_for("validation", val_data),
        np.arange(val_data.num_windows),
        cfg,
        zone_w,
        shuffle=False,
    )
    model = compile_model(cfg, spe * epochs)
    log.info(
        "%s seed %d: %d params, %d train windows, %d steps/epoch x %d epochs",
        cfg["name"],
        seed,
        model.count_params(),
        len(windows),
        spe,
        epochs,
    )
    paths["checkpoints"].mkdir(parents=True, exist_ok=True)
    paths["results"].mkdir(parents=True, exist_ok=True)
    guard = TimeGuard(cfg["max_hours"], session_start)
    callbacks = [
        SyncEpoch(train_ds),
        guard,
        keras.callbacks.ModelCheckpoint(
            str(paths["best_model"]),
            monitor=es["monitor"],
            mode=es["mode"],
            save_best_only=True,
            initial_value_threshold=best_logged_value(paths["history"], es["monitor"], es["mode"]),
        ),
        keras.callbacks.EarlyStopping(
            monitor=es["monitor"], mode=es["mode"], patience=es["patience"]
        ),
        keras.callbacks.BackupAndRestore(str(paths["backup"]), delete_checkpoint=False),
        keras.callbacks.CSVLogger(str(paths["history"]), append=True),
    ]
    history = model.fit(
        train_ds, validation_data=val_ds, epochs=epochs, callbacks=callbacks, verbose=2
    )
    manifest = {
        "stage": "model1_v3",
        "name": cfg["name"],
        "seed": seed,
        "commit": commit_sha(),
        "config": cfg,
        "device": device_info,
        "params": model.count_params(),
        "train_windows": len(windows),
        "zone_class_weights": zone_w,
        "steps_per_epoch": spe,
        "epochs_this_session": len(history.history.get("loss", [])),
        "started": started,
        "finished": utc_now(),
        "seconds": round(time.time() - start, 1),
    }
    if guard.stopped:
        manifest["status"] = "interrupted"
        write_json(paths["results"] / "manifest.json", manifest)
        log.warning("seed %d interrupted by the time guard; rerun to resume.", seed)
        return manifest
    shutil.rmtree(paths["backup"], ignore_errors=True)
    best = keras.saving.load_model(paths["best_model"], compile=False)
    val_preds = predict_sentences(best, val_data, cfg)
    threshold, _ = tune_threshold(np.asarray(val_data["importance"]), val_preds["importance"])
    zone_thresholds = calibrate_zones(val_data, val_preds, threshold, cfg)
    test_data = NotesSet.load(processed / "test")
    test_preds = predict_sentences(best, test_data, cfg)
    write_jsonl(
        paths["results"] / "test_predictions.jsonl",
        prediction_records(test_data, test_preds, zone_thresholds),
    )
    metrics = {
        "seed": seed,
        "params": best.count_params(),
        "threshold": threshold,
        "zone_thresholds": zone_thresholds,
        "validation": sentence_metrics(val_data, val_preds, threshold, zone_thresholds),
        "test": sentence_metrics(test_data, test_preds, threshold, zone_thresholds),
    }
    write_json(paths["results"] / "metrics.json", metrics)
    manifest.update(status="complete", seconds=round(time.time() - start, 1))
    write_json(paths["results"] / "manifest.json", manifest)
    log.info("%s seed %d done: %s", cfg["name"], seed, metrics["validation"]["all"])
    return manifest


def prediction_records(
    dataset: NotesSet,
    preds: dict[str, np.ndarray],
    zone_thresholds: dict[str, float] | None = None,
):
    """One record per meeting: sentence IDs, importance probabilities and zone predictions."""
    starts, lengths = np.asarray(dataset["note_start"]), np.asarray(dataset["note_length"])
    for note_id, s, n in zip(dataset.note_ids, starts, lengths, strict=True):
        rows = slice(int(s), int(s) + int(n))
        yield {
            "note_id": note_id,
            "sentence_ids": np.asarray(dataset["sentence_id"][rows]).tolist(),
            "importance": [round(float(p), 6) for p in preds["importance"][rows]],
            "zone": [ZONES[z] for z in assign_zones(preds["zone"][rows], zone_thresholds)],
        }


def deep_merge(base: dict[str, Any], override: dict[str, Any]) -> dict[str, Any]:
    """Return a new dict with ``override`` merged recursively into ``base``.

    Nested dicts are merged key by key; any other value in ``override`` replaces the base value.
    """
    merged = dict(base)
    for key, value in override.items():
        if isinstance(value, dict) and isinstance(merged.get(key), dict):
            merged[key] = deep_merge(merged[key], value)
        else:
            merged[key] = value
    return merged

## 4. Runs
E4 needs a passed Qwen gate (Notebook 1); without it only E1 runs. Each run resumes from its
backup if a session was cut, and is skipped once its `metrics.json` exists.

In [ ]:
def run_config(name):
    cfg = deep_merge(CONFIG["base"], CONFIG["variants"][name])
    cfg["name"] = name
    cfg["paths"] = {
        "corpus_dir": str(CORPUS),
        "processed_dir": str(PROCESSED),
        "teacher_dir": None if TEACHER_DIR is None else str(TEACHER_DIR),
        "init_checkpoint": None if MLM_ENCODER is None else str(MLM_ENCODER),
        "output_dir": str(CKPT / name),
        "results_dir": str(RESULTS),
    }
    return cfg


gate_ok = bool(GATE and GATE.get("passed"))
STATUS = {}
for name, seed in CONFIG["runs"]:
    key = f"{name}/seed_{seed}"
    cfg = run_config(name)
    if (run_paths(cfg, seed)["results"] / "metrics.json").exists():
        STATUS[key] = "complete"
        continue
    if "qwen" in cfg.get("teachers", []) and not gate_ok:
        STATUS[key] = "skipped: Qwen did not pass the quality gate (Notebook 1)"
        continue
    if hours_left() < CONFIG["min_hours_to_start"]:
        STATUS[key] = "not started: not enough time left (rerun to resume)"
        continue
    manifest = train_seed(cfg, seed, SESSION_START)
    STATUS[key] = manifest["status"]
    keras.backend.clear_session()
print(json.dumps(STATUS, indent=2))

## 5. Preview and manifest
The repo's `src/model1_notes/evaluate.py` builds the full table (baselines, teachers, AMI ROUGE, latency) from these outputs.

In [ ]:
cols = [("AMI imp F1", "ami", "importance_f1"), ("AMI PR-AUC", "ami", "pr_auc"),
        ("AMI dec F1", "ami", "decision_f1"), ("AMI act F1", "ami", "action_f1"),
        ("Synth imp F1", "synthetic", "importance_f1"), ("Synth zone F1", "synthetic", "zone_macro_f1"),
        ("Synth HN FP", "synthetic", "hard_negative_fp"), ("Spans F1", "spans", "span_f1")]
lines = ["| Run | Seed | Threshold | " + " | ".join(c[0] for c in cols) + " |",
         "|" + "---|" * (3 + len(cols))]
complete = {}
for name, seed in CONFIG["runs"]:
    path = RESULTS / name / f"seed_{seed}" / "metrics.json"
    if not path.exists():
        continue
    complete.setdefault(name, []).append(seed)
    m = json.loads(path.read_text())
    cells = [str((m["test"].get(group) or {}).get(key, "–")) for _, group, key in cols]
    lines.append(f"| {name} | {seed} | {m['threshold']} | " + " | ".join(cells) + " |")
preview = "# Model 1 v3 preview (test, gold labels)\n\n" + "\n".join(lines) + "\n"
(RESULTS / "tables").mkdir(parents=True, exist_ok=True)
(RESULTS / "tables" / "model1_v3_preview.md").write_text(preview)
print(preview)
json_dump(WORK / f"manifest_{STAGE}.json", {
    "stage": STAGE,
    "smoke": SMOKE,
    "config": CONFIG,
    "model": CONFIG["base"]["model"],
    "gate": GATE,
    "status": STATUS,
    "complete_seeds": complete,
    "finished_all": all(v == "complete" for v in STATUS.values()),
    "hours": round((time.time() - SESSION_START) / 3600, 3),
})